# Concert Program Structured Data with LLM

Extracts place, date, presenting institution, patrons/sponsors, composers and works performed, and performers (with instrument or vocal part) from OCR'd concert program PDFs, tagged with the source page number and merged with the manifest CSV's metadata.

- Notebook by Daniel Russo-Batterham, Charlie Cross, Richard Freedman, Miles Fagan.
- Adapted for concert programs, August 2026.

## 0.  Set up File Paths and Imports

In [ ]:
# initial imports of libraries

from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document

from typing import Optional, List

from langchain_core.prompts import ChatPromptTemplate

import getpass
import os
import csv
import json

from pydantic import BaseModel, Field

async def loadPDF(filepath: str) -> list:
    """Load a PDF and return one langchain Document per page, in page order."""
    loader = PyPDFLoader(filepath)
    pages = []
    async for page in loader.alazy_load():
        pages.append(page)
    return pages

def build_marked_up_text(pages: list) -> str:
    """Join per-page Documents into one string, with a literal page marker before each
    page's text. The LLM is asked to copy the nearest preceding marker into page_number,
    rather than compute or guess a page number itself."""
    parts = [f"--- PAGE {i} ---\n{page.page_content}" for i, page in enumerate(pages, start=1)]
    return "\n\n".join(parts)

In [ ]:
# folder containing all concert program PDFs, and the manifest describing them
pdf_dir = Path("sources")
manifest_path = pdf_dir / "manifest.csv"

# where structured output gets written
output_dir = Path("Structured Data")
output_dir.mkdir(exist_ok=True)

# for a quick test run, list a few filenames here (e.g. ["UDC20260028-10.pdf"]);
# leave empty to process every file in the manifest
sample_filenames: List[str] = ["UDC20260028-10.pdf", "UDC20260028-13.pdf", "UDC20260028-29.pdf"]

# cap on how many NOT-YET-PROCESSED files to extract in a single run (e.g. 10 at a time,
# re-running to work through the manifest in chunks); set to None to process all pending files
batch_size: Optional[int] = 10

In [ ]:
# load the manifest: filename -> {contents, date, organization}
manifest = {}
with open(manifest_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        manifest[row["Filename"]] = {
            "contents": row.get("Contents"),
            "date": row.get("Date"),
            "organization": row.get("Organization"),
        }

print(f"Loaded manifest for {len(manifest)} files")

### Select LLM and Provide API Key

- You can choose your preferred LLM here. Make sure to have the necessary API keys set up in your environment.



In [ ]:
# set up LLM model. Prompt for OpenAI key if not set in environment.  Just click "return" after you enter the key in the box

if not os.environ.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

from langchain.chat_models import init_chat_model

llm = init_chat_model("gpt-5", model_provider="openai")

## 1.  Setting up Classes for Pydantic

For concert programs we care about six kinds of item: the venue, the date, the presenting institution/organization, patrons and sponsors, the composers and works performed, and the musicians and other performers (with their instrument or vocal part). Each item is its own Pydantic class, and every item carries a `page_number`, a verbatim `source_text` excerpt, and `review_flags` for anything inferred or uncertain — so every extracted fact can be traced back to exactly where it was found and how confident we are in it.

In [ ]:
# Pydantic classes for concert-program structured extraction.
# Every record carries page_number + source_text (verbatim excerpt) + review_flags,
# so each fact can be traced back to where it was found and how confident we are in it.

class Venue(BaseModel):
    """A place where the concert took place (city, hall, building)."""
    page_number: int = Field(description="Page number copied from the nearest preceding '--- PAGE n ---' marker.")
    source_text: str = Field(description="Verbatim excerpt of the text supporting this record.")
    review_flags: List[str] = Field(default_factory=list, description="Reasons to flag this record for manual review, if any.")
    place: str = Field(description="The place of performance (city, hall, building) as it appears in the text.")

class PerformanceDate(BaseModel):
    """A date on which the concert took place."""
    page_number: int = Field(description="Page number copied from the nearest preceding '--- PAGE n ---' marker.")
    source_text: str = Field(description="Verbatim excerpt of the text supporting this record.")
    review_flags: List[str] = Field(default_factory=list, description="Reasons to flag this record for manual review, if any.")
    date_text: str = Field(description="The date of performance, verbatim as printed.")

class OrganizationMention(BaseModel):
    """An institution or organization presenting or performing the concert."""
    page_number: int = Field(description="Page number copied from the nearest preceding '--- PAGE n ---' marker.")
    source_text: str = Field(description="Verbatim excerpt of the text supporting this record.")
    review_flags: List[str] = Field(default_factory=list, description="Reasons to flag this record for manual review, if any.")
    name: str = Field(description="The organization's name as it appears in the text.")
    role: Optional[str] = Field(default=None, description="The organization's role, e.g. 'presenting organization', 'choir', 'orchestra'.")

class Patron(BaseModel):
    """A patron or sponsor named in the program, distinct from the performing organization."""
    page_number: int = Field(description="Page number copied from the nearest preceding '--- PAGE n ---' marker.")
    source_text: str = Field(description="Verbatim excerpt of the text supporting this record.")
    review_flags: List[str] = Field(default_factory=list, description="Reasons to flag this record for manual review, if any.")
    name: str = Field(description="The patron or sponsor's name as it appears in the text.")
    role: Optional[str] = Field(default=None, description="Their role, e.g. Patron, Vice-Patron, Sponsor, Donor, President.")

class WorkPerformed(BaseModel):
    """A composed work performed at the concert."""
    page_number: int = Field(description="Page number copied from the nearest preceding '--- PAGE n ---' marker.")
    source_text: str = Field(description="Verbatim excerpt of the text supporting this record.")
    review_flags: List[str] = Field(default_factory=list, description="Reasons to flag this record for manual review, if any.")
    composer: Optional[str] = Field(default=None, description="The work's composer, if given.")
    title: str = Field(description="The title of the work performed.")
    movement_or_selection: Optional[str] = Field(default=None, description="A specific movement or selection from the work, if indicated.")

class Performer(BaseModel):
    """A musician or other performer, and what they played, sang, or their role."""
    page_number: int = Field(description="Page number copied from the nearest preceding '--- PAGE n ---' marker.")
    source_text: str = Field(description="Verbatim excerpt of the text supporting this record.")
    review_flags: List[str] = Field(default_factory=list, description="Reasons to flag this record for manual review, if any.")
    name: str = Field(description="The performer's name as it appears in the text.")
    part_or_instrument: Optional[str] = Field(default=None, description="The instrument played, vocal part sung, or role (e.g. 'soprano', 'violin', 'conductor', 'accompanist').")
    associated_work: Optional[str] = Field(default=None, description="The title of the work they are credited on, if clear.")

class ConcertProgramExtraction(BaseModel):
    """Everything extracted from one concert program PDF."""
    venues: List[Venue] = Field(default_factory=list)
    dates: List[PerformanceDate] = Field(default_factory=list)
    organizations: List[OrganizationMention] = Field(default_factory=list)
    patrons: List[Patron] = Field(default_factory=list)
    works: List[WorkPerformed] = Field(default_factory=list)
    performers: List[Performer] = Field(default_factory=list)

In [ ]:
def flatten_extraction(filename: str, meta: dict, extraction_dict: dict) -> List[dict]:
    """Turn one concert's ConcertProgramExtraction (as a plain dict) into flat item records,
    each stamped with its record_type and the file's manifest metadata."""
    items = []
    for record_type, key in [
        ("venue", "venues"),
        ("date", "dates"),
        ("organization", "organizations"),
        ("patron", "patrons"),
        ("work", "works"),
        ("performer", "performers"),
    ]:
        for record in extraction_dict[key]:
            items.append({
                "record_type": record_type,
                "filename": filename,
                "manifest_contents": meta["contents"],
                "manifest_date": meta["date"],
                "manifest_organization": meta["organization"],
                **record,
            })
    return items

## 2.  Bind the Schema to the LLM

`ConcertProgramExtraction` bundles all six item lists for one concert program. We bind it to the LLM's structured-output mode so a single call returns validated Pydantic objects rather than raw JSON.

In [ ]:
structured_llm = llm.with_structured_output(ConcertProgramExtraction)

## 3.  The System Prompt

In [ ]:
# Common part/role abbreviations found in these historical programs -- mostly Italian ordinal
# conventions for orchestral sections. Add more entries here as you find them across the corpus;
# they're automatically folded into the system prompt below, no prompt-editing required.
#
# "Imi."/"I mi" = Primi ("First"), "IIdi." = Secondi ("Second"), "IIIzi." = Terzi ("Third")
role_glossary = {
    "Violini Imi.": "First Violin",
    "Violini IIdi.": "Second Violin",
    "Viole": "Viola",
    "Violoncelli": "Violoncello (Cello)",
    "Contrabassi": "Double Bass",
    "Flauti Imi.": "First Flute",
    "Flauti IIdi.": "Second Flute",
    "Oboi": "Oboe",
    "Clarinetti": "Clarinet",
    "Fagotti": "Bassoon",
    "Corni": "Horn",
    "Trombe": "Trumpet",
    "Tromboni": "Trombone",
    "Timpani": "Timpani",
}

In [ ]:
# system and human prompts for extracting concert-program data -- may need adjustment for different printers/OCR quality

role_glossary_text = "\n".join(f'- "{k}" = {v}' for k, v in role_glossary.items())

system_prompt = f"""
You are a musicologist and archivist specializing in 19th- and 20th-century concert programs.
You will be given the full text of one concert program, extracted via OCR from a PDF. The text
has literal page markers of the form "--- PAGE n ---" inserted before each page's content.

Your job is to extract every mention of the following, across all pages:
- Place of performance (city, hall, building)
- Date of performance
- The institution or organization presenting/performing the concert (e.g. a choral society,
  orchestra, or academy)
- Patrons and sponsors -- people or bodies named as patron, vice-patron, sponsor, donor, or
  similar honorary/financial role. These are usually distinct from the performing organization
  itself (e.g. a Governor or Lord Mayor listed as "Patron" of a concert given by the
  "Melbourne Liedertafel").
- Composers and the works they performed
- Musicians and other performers, and what instrument they played or vocal part they sang
  (or their role, such as conductor or accompanist)

Rules:
- For every item you extract, set page_number to the number copied from the nearest preceding
  "--- PAGE n ---" marker. Never guess or compute a page number -- only copy it from a marker.
- For every item, set source_text to a short verbatim excerpt of the OCR text that supports it.
  Do not paraphrase or correct the wording in source_text.
- Name and title fields (composer, title, and every person/organization name) must always hold
  the text exactly as printed/OCR'd -- never silently correct or normalize a spelling. If the
  OCR is garbled but you can confidently guess the correct reading (e.g. "Geethoven" for
  "Beethoven"), keep the garbled form in the field itself and add a review_flags note giving
  your best-guess correction. Only use null when the text is too garbled to make any confident
  guess at all, and flag it as unrecoverable.
- A performer's part_or_instrument is different: it is a controlled-vocabulary field, so
  normalize it to a clear, standard English description rather than reproducing the printed
  abbreviation. Many of these programs abbreviate orchestral sections using Italian ordinal
  conventions: "Imi."/"I mi" = Primi ("First"), "IIdi." = Secondi ("Second"), "IIIzi." = Terzi
  ("Third"). For example "Violini Imi." means First Violin, "Corni IIdi." means Second Horn.
  Known examples from this corpus:
{role_glossary_text}
  Apply the same ordinal pattern to any instrument section you recognize it on, even if it is
  not in the list above. If you encounter a role/instrument abbreviation you do not recognize at
  all, keep it as printed in part_or_instrument and add a review_flags note rather than guessing.
- Some pages are not part of the concert program itself (e.g. a travel itinerary, an
  institutional history/prospectus, or garbled/illegible OCR noise). Such pages should simply
  yield no items -- do not invent a place, date, work, or performer that isn't there.
- Whenever you infer a value, expand an abbreviated name, guess at OCR-garbled text, or make an
  ambiguous attribution (for example, a role caption that isn't clearly tied to one name), add a
  short reason to that item's review_flags. Prefer leaving a field null and flagging it over
  fabricating a value.
- If a work lists several movements or several composers, emit one work item per distinct work.
- If a performer is clearly credited on a specific work, set that performer's associated_work to
  that work's title.
"""

human_prompt = """
Given the following concert program (with page markers), extract every venue, date,
organization, patron/sponsor, work performed, and performer as a structured
ConcertProgramExtraction object.
"""

## 4.  Extract Data from Each Concert Program

Each PDF is short enough to send in a single LLM call: we join its pages into one string with literal `--- PAGE n ---` markers and let the model copy the right marker into each item's `page_number`, which also lets it correlate a work on one page with performers listed on the next. Since there's only one call per document now (no scene-by-scene chunking), we don't need a LangGraph graph -- a plain function does the job.

This step also **resumes** rather than starting over: it loads any previously saved `concert_program_by_file.json` as a cache and only calls the LLM for filenames not already in it. So if you add new rows to `manifest.csv` later, re-running this notebook only pays for the new files -- it won't re-extract ones already done. To force a specific file to be redone (e.g. after a prompt change), add its filename to `force_reprocess_filenames` below.

In [ ]:
# don't edit these

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "Context:\n{context}\n\nQuestion:\n{question}")
])

async def extract_concert(pdf_path: Path) -> ConcertProgramExtraction:
    """Load one concert program PDF and run structured extraction on it in a single LLM call."""
    pages = await loadPDF(str(pdf_path))
    marked_up_text = build_marked_up_text(pages)
    message = prompt.invoke({"question": human_prompt, "context": marked_up_text})
    return structured_llm.invoke(message)

In [ ]:
# resume support: reuse any previously extracted results so re-running after adding new
# files to the manifest doesn't re-call the LLM (and re-pay for) files already processed
by_file_json_path = output_dir / "concert_program_by_file.json"
if by_file_json_path.exists():
    with open(by_file_json_path, encoding="utf-8") as f:
        by_file: dict = json.load(f)
    print(f"Loaded {len(by_file)} previously processed files from {by_file_json_path}")
else:
    by_file: dict = {}

# list filenames here to force them to be re-extracted even if already cached (e.g. after a
# prompt change)
force_reprocess_filenames: List[str] = []

filenames_to_process = sample_filenames if sample_filenames else list(manifest.keys())
pending_filenames = [
    f for f in filenames_to_process
    if f not in by_file or f in force_reprocess_filenames
]
filenames_to_extract = pending_filenames if batch_size is None else pending_filenames[:batch_size]

print(f"{len(filenames_to_process) - len(pending_filenames)} already cached, "
      f"{len(pending_filenames)} pending, extracting {len(filenames_to_extract)} this run")

for filename in filenames_to_extract:
    meta = manifest.get(filename)
    if meta is None:
        print(f"Skipping {filename}: not found in manifest")
        continue

    pdf_path = pdf_dir / filename
    if not pdf_path.exists():
        print(f"Skipping {filename}: PDF not found in {pdf_dir}")
        continue

    try:
        extraction = await extract_concert(pdf_path)
    except Exception as e:
        print(f"Failed to extract {filename}: {e}")
        continue

    by_file[filename] = extraction.model_dump()
    print(f"Extracted {filename}")

print(f"by_file now has {len(by_file)} processed files total")

## 5.  Output

We write the extracted data three ways, always rebuilt from the *full* accumulated `by_file` cache (everything ever processed, not just this run's new extractions):

- `concert_program_items.json` -- a flat JSON array, one object per extracted item (venue, date, organization, patron, work, or performer), each carrying its `record_type`, source `filename`, the manifest metadata (Contents/Date/Organization), and the `page_number` it was found on.
- `concert_program_by_file.json` -- the same data grouped by source file. This also doubles as the resume cache read by the extraction step above.
- `concert_program_items.csv` -- a flat CSV mirror of the item list, for spreadsheet use.

In [ ]:
# flatten every processed file (cached + newly extracted) into item records
all_items: List[dict] = []
for filename, extraction_dict in by_file.items():
    meta = manifest.get(filename)
    if meta is None:
        continue  # file no longer listed in the manifest
    all_items.extend(flatten_extraction(filename, meta, extraction_dict))

# flat JSON: one record per extracted item (venue, date, organization, patron, work, performer)
flat_json_path = output_dir / "concert_program_items.json"
with open(flat_json_path, "w", encoding="utf-8") as f:
    json.dump(all_items, f, indent=2, ensure_ascii=False)

# nested JSON: grouped by source file -- also the resume cache read by the extraction step
with open(by_file_json_path, "w", encoding="utf-8") as f:
    json.dump(by_file, f, indent=2, ensure_ascii=False)

# flat CSV mirror of the item list
csv_path = output_dir / "concert_program_items.csv"
fieldnames = sorted({key for item in all_items for key in item.keys()})
with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for item in all_items:
        row = dict(item)
        if isinstance(row.get("review_flags"), list):
            row["review_flags"] = "; ".join(row["review_flags"])
        writer.writerow(row)

print(f"Wrote {len(all_items)} items to {flat_json_path}")
print(f"Wrote {len(by_file)} files to {by_file_json_path}")
print(f"Wrote {len(all_items)} rows to {csv_path}")